# Portfolio Opdracht 3: 


# 1. Inleiding
Je bent aangenomen als junior AI engineer bij VisionWear AI, een technologiebedrijf dat
kunstmatige intelligentie ontwikkelt voor de mode-industrie. Het bedrijf werkt samen met
webshops, modehuizen en online kledingplatforms om slimme AI-systemen te bouwen die
kleding automatisch kunnen herkennen, beschrijven en zelfs nieuwe ontwerpen kunnen
genereren.
VisionWear AI wil een volledig intelligent modesysteem ontwikkelen dat:
1. kledingstukken automatisch kan detecteren in afbeeldingen,
2. modieuze productbeschrijvingen kan schrijven,
3. en nieuwe kledingafbeeldingen kan genereren op basis van tekstuele beschrijvingen.
Om dit systeem te ontwikkelen heeft het bedrijf een grote dataset verzameld met
modefoto’s, labels en beschrijvingen. Jullie taak is om als data science-team verschillende AI-
modellen te bouwen die bijdragen aan deze innovatieve modepipeline.
De opdracht bestaat uit drie delen:
In deel 1 train je een model om kledingstukken en modeaccessoires in afbeeldingen te
identificeren. Jouw model moet de categorie van de artikel(en) voorspellen, evenals de
coördinaten van de locatie ervan binnen de afbeelding. Voor elke afbeelding kunnen een of
1
meer relevante items worden geïdentificeerd en hun locatie binnen de afbeelding kan
worden bepaald. Je neemt met een team deel aan een Kaggle-wedstrijd waarbij je het
opneemt tegen andere teams uit de opleiding Applied Data Science & AI. De team(s) die het
hoogst eindigt in de wedstrijd krijgt 5 bonuspunten bovenop het aantal behaalde punten
met de opdracht.
In deel 2 train je een model om tekstbijschriften (in het Engels) te genereren die de
kledingstukken beschrijven met behulp van relevante modetermen.
In deel 3 ga je een model fine-tunen om afbeeldingen van kleding te genereren op basis van
tekstbeschrijvingen (in het Engels

# Vereisten notebook
Notebooks die niet voldoen aan onderstaande voorwaarden worden niet nagekeken. In dit
geval moet je gebruik maken van de herkansing om een cijfer voor deze opdracht te krijgen.

 Lever één net en duidelijk gestructureerd .html-versie van je notebook in op
Brightspace.

 Zorg ervoor dat alle codecellen zijn uitgevoerd en dat de verwachte output zichtbaar
is, zonder foutmeldingen of rommelige output!

 Voeg een link toe naar het .ipynb-notebook op Github.

 Structureer het notebook met markdown cellen en nummer de hoofdstukken en
paragraven.

 Gebruik markdown cellen voor tekst en code cellen voor code.

 Gebruik zoveel mogelijk zelf-gedefinieerde functies en bij voorkeur OOP.

 De code voldoet aan PEP8 inclusief “comments”.

 Alle groepsgenoten begrijpen alle code en de teksten die worden ingeleverd en zijn in
staat deze toe te lichten als daarom wordt gevraagd.

 Refereer voor zowel de tekst als voor de code op de juiste wijze aan gebruikte
bronnen (zie bijlage 1).

Andere vereisten
Portfolio's die niet aan deze eisen voldoen, worden niet beoordeeld. In dit geval moet je
gebruik maken van de herkansing om een cijfer voor deze opdracht te krijgen.

 Jouw groep moet minimaal één submission op Kaggle hebben.

 Je dient een minimumscore op Kaggle te halen. Als je score slechter is, zal je gebruik
moeten maken van de herkansing.

o Voor Portfolioopdracht 2 is de minimumscore een Mean Average Precision
(mAP) van 0,7 (70%) bij een drempelwaarde van 0,5 (50%) Intersection over
Union (IoU). Of mAP @ 0,5 moet 0,7 of hoger.

 Jouw inlevering op Kaggle moet het resultaat zijn van je eigen getrainde modellen.
Als je resultaten van een ander model inlevert, OF als je de juiste labels online vindt
en deze inlevert, wordt je inlevering gediskwalificeerd.

In [1]:
import os

# Omdat de cuda net geupdate is, werd alles opeens instabiel,
# en aangezien tensorflow en pytorch een hekel aan elkaar hebben, 
# moest ik deze workaround gebruiken,
# en gwn memory reserven voor beide frameworks
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Dank aan deze stackoverflow post die uitlegde ik fatsoenlijk mn cuda libraries moest linken,
# BRON: https://stackoverflow.com/questions/13428910/how-to-set-the-environment-variable-ld-library-path-in-linux 
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:/opt/cuda/targets/x86_64-linux/lib'

import nlpaug.augmenter.word as naw
import spacy

# Memory limit fix voor tensorflow, zodat er genoeg VRAM overblijft voor pytorch
# Dankjewel aan Slingacademy voor deze uitleg en fix, 
# want zonder deze fix was het onmogelijk om beide frameworks te gebruiken op dezelfde GPU
# Omdat Tensforflow en Pytorch zo hungry is voor VRAM
# BRON: https://www.slingacademy.com/article/optimizing-memory-allocation-with-tensorflow-config/#preallocating-gpu-memory 
tensorflow_vram_limit = 4048
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:

        tf.config.set_logical_device_configuration(
            gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=tensorflow_vram_limit)]
        )
        print(f"TensorFlow beperkt tot {tensorflow_vram_limit}")
    except: pass

import torch
from transformers import (
    pipeline, 
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    AutoConfig,
    DataCollatorWithPadding,
    AutoModelForMaskedLM
)

print(f"Kan Pytorch bij de rest van de 12GB VRAM: {torch.cuda.is_available()}")

I0000 00:00:1779704138.469557   15836 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779704139.745537   15836 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow beperkt tot 4048
Kan Pytorch bij de rest van de 12GB VRAM: True


In [2]:
# Inladen van libraries
# iets in de volgorde zorgt ervoor dat er geen problemen zijn met cuda libraries,
# dus laat de volgorde zo staan AUB.
import os
import re
import gc
import random
import time
from pathlib import Path
from collections import Counter
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import IPython.display as ipd
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack

from tqdm.auto import tqdm

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.utils import class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

from tensorflow.keras import layers, models, optimizers, callbacks, regularizers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import plot_model

from datasets import Dataset

from torchview import draw_graph 

In [3]:
# Hier maken we de variables in van de file paths,
# aangezien we te maken hebben m​et verschillende files en locations.

# Met de base location van de Notebook
BASE_DIR = Path.cwd()

# De directories van de Wav Files
TRAIN_AUDIO_DIR = BASE_DIR / "Train"
TEST_AUDIO_DIR = BASE_DIR / "Test"

# En de CSV Files
TRAIN_CSV = BASE_DIR / "train.csv"
TEST_CSV = BASE_DIR / "test.csv"
SAMPLE_SUBMISSION = BASE_DIR / "sample_solution.csv"

# En dan laden we de data in van de CSV's
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION)

FileNotFoundError: [Errno 2] No such file or directory: '/home/beef/Downloads/karaoke-k-klassificatie-2026/Deep-Learning-3/train.csv'

# Opdracht 1: Exploratieve Data Analyse
 Ieder teamlid maakt een account op Kaggle aan en vormt een team dat deelneemt
aan deze competitie: Transforming Fashion 2026 | Kaggle.
 Noteer je teamnaam, jullie namen en alle bijhorende Kaggle gebruikersnamen in het
notebook.
 Voer een Exploratieve Data Analyse (EDA) uit:
4
o Visualiseer enkele afbeeldingen en hun bounding-boxes.
o Bestudeer de verschillende categorieën items en hoe ze in de afbeeldingen
verschijnen.
 Geef een paar afbeeldingen uit de image-captioning dataset weer, samen met hun
bijschriften.
 Welke items zijn gemeenschappelijk voor zowel de captioning-dataset als de
objectdetectiedataset?
 Beschrijf wat de belangrijkste bevindingen zijn van de EDA

# Opdracht 2: Objectdetectie
 In deze deelopdracht werk je uitsluitend met de images en labels uit de data voor
deel 1. Bouw en train een objectdetectie model om de tien klassen kleding en
modeaccessoires in de dataset te detecteren en hun locaties te voorspellen met
behulp van bounding-boxes.
 Beschrijf in je eigen woorden hoe je model voorspellingen doet.
 Beschrijf hoe je de afbeeldingen voorbewerkt.
 Beschrijf in detail welke stappen, indien aanwezig, je onderneemt voor feature
engineering.
 Onderbouw je keuzes van de hyperparameters, de keuze van de optimizer en het
aantal trainingsepochs.
 Beschrijf de lossfunctie voor dit model in je eigen woorden en hoe deze werk

# Opdracht 3: Ondertiteling van afbeeldingen
 In deze deelopdracht werk je met de H&M images en captions dataset.
 Fine-tune een model om Engelse tekstbijschriften te genereren op basis van
afbeeldingen van kleding.
 Beschrijf in je eigen woorden hoe het model voorspellingen doet, welke lossfunctie
het gebruikt en hoe het wordt getraind.
 Onderbouw je keuzes van de hyperparameters, de keuze van de optimizer en het
aantal trainingsepochs

# Opdracht 4: Beeldgeneratie
 In deze deelopdracht werk je met de H&M images en captions dataset.
 Voor deze opdracht kies je een voorgetraind model van Hugging Face en fine-tune dit
op de afbeeldingen en tekstdataset.
 Geef de naam op van de modelarchitectuur die je hebt gekozen en de dataset
waarop deze vooraf is getraind.
 Je model moet een Engels tekstbijschrift als input gebruiken en een afbeelding als
output maken.
 Beschrijf in detail de stappen die worden genomen bij het finetunen van dit vooraf
getraind model.
5
 Beschrijf de componenten van dit model.
 Beschrijf in je eigen woorden hoe het model afbeeldingen maakt en welke lossfunctie
het gebruikt.
 Onderbouw je keuzes van de hyperparameters, de keuze van de optimizer en het
aantal trainingsepoch

# Opdracht 5: Implementatie
 Maak een eenvoudige app die een foto als input gebruikt, kledingstukken en
modeaccessoires detecteert en hun locatie geeft, en tekstbeschrijvingen genereert
van elk kledingstuk in de afbeelding. Gebruik de modellen die je in de voorgaande
opdrachten hebt getraind/gefinetuned.
 De app gebruikt een foto als input en voorspelt de locatie van elk afzonderlijk
kledingstuk en modeaccessoire. Vervolgens worden de individuele
kledingstukken/mode-items voorzien van bijschriften. De bijschriften moeten voor
elk afzonderlijk gedetecteerd item afzonderlijk worden gegenereerd.
 Let op: Je hoeft deze app niet te deployen. Je hoeft alleen maar de modellen in jouw
notebook uit te voeren en de voorspellingen op de juiste manier te combineren.
 Geef een samenvatting van de uitkomsten van het modelleren.
o Geef een beknopt overzicht van de resultaten.
o (Voor deel 1) Toon je scores op Kaggle en laat zien wat de resultaten waren
van je verbeteringen op je score op Kaggle.
o (Voor deel 2 & 3) Geef voorbeelden van je getrainde modellen in actie en
evalueer hun prestaties.
o Geef een voorbeeld/voorbeelden van hoe je app kleding en modeaccessoires
detecteert en geschikte bijschriften genereert voor elk gedetecteerd ite

# Opdracht 6: Conclusie en aanbevelingen
 Beschrijf het modelleringsproces voor deze opdracht. Wat waren de uitdagingen
daarmee?
 Wat zou je aanbevolen gebruik zijn van de modellen die je hebt getraind?
 Wat zouden je aanbevelingen zijn met betrekking tot het soort afbeeldingen/data
waarop je deze modellen kunt gebruiken?
 Zijn de resultaten van je modellen accuraat/betrouwbaar? Wat kan er gedaan
worden om deze te verbeteren?